# Notebook 14 — StratLake Campaign Evidence Review Pack and Governance Audit

Source-safe raw development draft for Notebook 14.

Notebook 14 follows Notebook 13 and reviews campaign artifacts, restored feature/QA artifacts, derived evidence review packs, promotion governance outputs, catalog/lineage surfaces, caveats, and runtime handoff summaries without claiming production readiness, strategy approval, promotion readiness, governance readiness, statistical significance, alpha validation, artifact completeness, split-metric completeness, source/runtime equivalence, or canonical status for notebook-generated summaries.


## 1. Install dependencies


In [ ]:
!pip install -q "pandas-market-calendars>=5.0"
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine


## 2. Runtime overrides

Uncomment only in an executed runtime copy. Keep committed source preview-only.


In [ ]:
import os

# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_governance_preflight"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "archive_restore_discovery"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_review_pack_build"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "governance_report_run"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "catalog_lineage_review"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_governance_full_review"

# os.environ["NOTEBOOK14_ALLOW_STRATLAKE_INIT"] = "true"
# os.environ["RUN_STRATLAKE_INIT"] = "true"

# os.environ["NOTEBOOK14_ALLOW_DRIVE_MOUNT"] = "true"
# os.environ["NOTEBOOK14_DRIVE_MOUNT_POINT"] = "/content/drive"
# os.environ["NOTEBOOK14_DRIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab"

# os.environ["NOTEBOOK14_ALLOW_ARCHIVE_RESTORE"] = "true"
# os.environ["RUN_ARCHIVE_RESTORE"] = "true"
# os.environ["NOTEBOOK14_RESTORE_ARCHIVE_ID"] = "notebook-session-001"
# os.environ["NOTEBOOK14_DRIVE_ARCHIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab/session_archives"
# os.environ["NOTEBOOK14_RESTORE_ARCHIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab/session_archives/notebook-session-001"
# os.environ["NOTEBOOK14_RESTORE_TARGET_ROOT"] = "/content/stratlake-notebooks-restored/notebook-session-001"
# os.environ["NOTEBOOK14_RESTORE_RUN_ID"] = "manual-smoke-001"
# os.environ["NOTEBOOK14_POST_RESTORE_INVENTORY_LIMIT"] = "200"

# os.environ["NOTEBOOK14_ALLOW_EVIDENCE_REVIEW"] = "true"
# os.environ["RUN_EVIDENCE_REVIEW_PACK_BUILD"] = "true"
# os.environ["RUN_EVIDENCE_REVIEW_PACK_VALIDATE"] = "true"
# os.environ["NOTEBOOK14_SELECTED_RUN_ID"] = "strategy_001"

# os.environ["NOTEBOOK14_ALLOW_GOVERNANCE_REPORT"] = "true"
# os.environ["RUN_PROMOTION_GOVERNANCE_REPORT"] = "true"

# os.environ["NOTEBOOK14_ALLOW_CATALOG_LINEAGE"] = "true"
# os.environ["RUN_CATALOG_LINEAGE_EXPORT"] = "true"

# os.environ["NOTEBOOK14_INSTALL_RESOLVER_WARNING"] = "true"


## 3. Imports and helpers


In [ ]:
import json, os, shutil, subprocess, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

INSTALL_CAVEATS = []
if os.environ.get("NOTEBOOK14_INSTALL_RESOLVER_WARNING", "false").lower() == "true":
    INSTALL_CAVEATS.append("Install resolver warning was observed during package installation; monitor dependency compatibility before relying on affected optional surfaces.")

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def display_df(df: pd.DataFrame, max_rows: int = 20) -> None:
    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))

def display_markdown(text: str) -> None:
    if display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def command_available(command: str) -> bool:
    return shutil.which(command) is not None

def tail_text(text: str | None, max_chars: int = 4000) -> str:
    return "" if not text else text[-max_chars:]

def safe_json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, datetime):
        return value.isoformat()
    return str(value)

def write_json(path: Path, data: Any) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(data, indent=2, default=safe_json_default), encoding="utf-8")
    return path

def write_dataframe_csv(path: Path, df: pd.DataFrame) -> Path:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)
    return path


## 4. Profiles, Drive bootstrap, and workspace paths


In [ ]:
NOTEBOOK14_TEST_PROFILE = os.environ.get("NOTEBOOK14_TEST_PROFILE", "evidence_governance_preview").strip() or "evidence_governance_preview"
PROFILE_MATRIX = {
    "evidence_governance_preview": dict(init=False, drive=False, restore=False, review=False, governance=False, lineage=False, checkpoint=False),
    "evidence_governance_preflight": dict(init=True, drive=False, restore=False, review=False, governance=False, lineage=False, checkpoint=False),
    "campaign_artifact_discovery": dict(init=False, drive=False, restore=False, review=False, governance=False, lineage=False, checkpoint=False),
    "archive_restore_discovery": dict(init=True, drive=True, restore=True, review=False, governance=False, lineage=False, checkpoint=False),
    "evidence_review_pack_build": dict(init=True, drive=False, restore=False, review=True, governance=False, lineage=False, checkpoint=False),
    "governance_report_run": dict(init=True, drive=False, restore=False, review=False, governance=True, lineage=False, checkpoint=False),
    "catalog_lineage_review": dict(init=True, drive=False, restore=False, review=False, governance=False, lineage=True, checkpoint=False),
    "evidence_governance_full_review": dict(init=True, drive=True, restore=True, review=True, governance=True, lineage=True, checkpoint=True),
}
if NOTEBOOK14_TEST_PROFILE not in PROFILE_MATRIX:
    raise ValueError(f"Unknown NOTEBOOK14_TEST_PROFILE={NOTEBOOK14_TEST_PROFILE!r}")
PROFILE = PROFILE_MATRIX[NOTEBOOK14_TEST_PROFILE]

def env_true(name: str) -> bool:
    return os.environ.get(name, "false").lower() == "true"

ALLOW_STRATLAKE_INIT = env_true("NOTEBOOK14_ALLOW_STRATLAKE_INIT")
ALLOW_DRIVE_MOUNT = env_true("NOTEBOOK14_ALLOW_DRIVE_MOUNT")
ALLOW_ARCHIVE_RESTORE = env_true("NOTEBOOK14_ALLOW_ARCHIVE_RESTORE")
ALLOW_EVIDENCE_REVIEW = env_true("NOTEBOOK14_ALLOW_EVIDENCE_REVIEW")
ALLOW_GOVERNANCE_REPORT = env_true("NOTEBOOK14_ALLOW_GOVERNANCE_REPORT")
ALLOW_CATALOG_LINEAGE = env_true("NOTEBOOK14_ALLOW_CATALOG_LINEAGE")
ALLOW_ARCHIVE_CHECKPOINT = env_true("NOTEBOOK14_ALLOW_ARCHIVE_CHECKPOINT")
RUN_STRATLAKE_INIT = PROFILE["init"] and ALLOW_STRATLAKE_INIT and env_true("RUN_STRATLAKE_INIT")
RUN_ARCHIVE_RESTORE = PROFILE["restore"] and ALLOW_ARCHIVE_RESTORE and env_true("RUN_ARCHIVE_RESTORE")
RUN_EVIDENCE_REVIEW_PACK_BUILD = PROFILE["review"] and ALLOW_EVIDENCE_REVIEW and env_true("RUN_EVIDENCE_REVIEW_PACK_BUILD")
RUN_EVIDENCE_REVIEW_PACK_VALIDATE = PROFILE["review"] and ALLOW_EVIDENCE_REVIEW and env_true("RUN_EVIDENCE_REVIEW_PACK_VALIDATE")
RUN_PROMOTION_GOVERNANCE_REPORT = PROFILE["governance"] and ALLOW_GOVERNANCE_REPORT and env_true("RUN_PROMOTION_GOVERNANCE_REPORT")
RUN_CATALOG_LINEAGE_EXPORT = PROFILE["lineage"] and ALLOW_CATALOG_LINEAGE and env_true("RUN_CATALOG_LINEAGE_EXPORT")
RUN_ARCHIVE_CHECKPOINT = PROFILE["checkpoint"] and ALLOW_ARCHIVE_CHECKPOINT and env_true("RUN_ARCHIVE_CHECKPOINT")

DRIVE_NEEDED = PROFILE["drive"] or RUN_ARCHIVE_RESTORE
DRIVE_MOUNT_POINT = Path(os.environ.get("NOTEBOOK14_DRIVE_MOUNT_POINT", "/content/drive"))
DRIVE_ROOT = Path(os.environ.get("NOTEBOOK14_DRIVE_ROOT", DRIVE_MOUNT_POINT / "MyDrive" / "stratlake-colab"))

def is_drive_mounted(path: Path = DRIVE_MOUNT_POINT) -> bool:
    return path.exists() and any(path.iterdir()) if path.exists() else False

drive_status = {
    "in_colab": IN_COLAB,
    "required": DRIVE_NEEDED,
    "allow_drive_mount": ALLOW_DRIVE_MOUNT,
    "mount_point": DRIVE_MOUNT_POINT.as_posix(),
    "drive_root": DRIVE_ROOT.as_posix(),
    "mounted_before": is_drive_mounted(),
    "attempted_mount": False,
    "mounted_after": is_drive_mounted(),
    "error": None,
}
if DRIVE_NEEDED and IN_COLAB and ALLOW_DRIVE_MOUNT and drive is not None and not is_drive_mounted():
    try:
        drive_status["attempted_mount"] = True
        drive.mount(DRIVE_MOUNT_POINT.as_posix())
    except Exception as exc:
        drive_status["error"] = f"{type(exc).__name__}: {exc}"
drive_status["mounted_after"] = is_drive_mounted()

CAVEATS = list(INSTALL_CAVEATS)
if DRIVE_NEEDED and not drive_status["mounted_after"]:
    CAVEATS.append("Drive-backed archive restore requested but Google Drive is not mounted or unavailable.")

DEFAULT_STRATLAKE_ROOT = Path("/content/stratlake-notebooks") if IN_COLAB else Path.cwd() / "stratlake-notebooks"
STRATLAKE_ROOT = Path(os.environ.get("NOTEBOOK14_STRATLAKE_ROOT", DEFAULT_STRATLAKE_ROOT)).resolve()
NOTEBOOKS_ROOT = STRATLAKE_ROOT / "notebooks"
CONFIG_ROOT = STRATLAKE_ROOT / "configs"
DOCS_ROOT = STRATLAKE_ROOT / "docs"
CONTRACTS_ROOT = STRATLAKE_ROOT / "contracts"
ARTIFACT_ROOT = STRATLAKE_ROOT / "artifacts"
NOTEBOOK14_RUNTIME_ROOT = ARTIFACT_ROOT / "_notebook_14_runtime"
CAMPAIGN_ARTIFACT_ROOT = ARTIFACT_ROOT
EVIDENCE_REVIEW_OUTPUT_DIR = ARTIFACT_ROOT / "_derived" / "evidence_review"
GOVERNANCE_OUTPUT_DIR = ARTIFACT_ROOT / "promotion_governance"
CATALOG_LINEAGE_OUTPUT_DIR = NOTEBOOK14_RUNTIME_ROOT / "catalog_lineage"
SUMMARY_OUTPUT_DIR = NOTEBOOK14_RUNTIME_ROOT / "summary"

RESTORE_ARCHIVE_ID = os.environ.get("NOTEBOOK14_RESTORE_ARCHIVE_ID", "").strip()
DRIVE_ARCHIVE_ROOT = Path(os.environ.get("NOTEBOOK14_DRIVE_ARCHIVE_ROOT", DRIVE_ROOT / "session_archives")) if (DRIVE_NEEDED or os.environ.get("NOTEBOOK14_DRIVE_ARCHIVE_ROOT")) else None
RESTORE_ARCHIVE_ROOT_OVERRIDE = os.environ.get("NOTEBOOK14_RESTORE_ARCHIVE_ROOT", "").strip()
RESTORE_ARCHIVE_ROOT = Path(RESTORE_ARCHIVE_ROOT_OVERRIDE).resolve() if RESTORE_ARCHIVE_ROOT_OVERRIDE else ((DRIVE_ARCHIVE_ROOT / RESTORE_ARCHIVE_ID).resolve() if DRIVE_ARCHIVE_ROOT is not None and RESTORE_ARCHIVE_ID else None)
RESTORE_TARGET_ROOT_OVERRIDE = os.environ.get("NOTEBOOK14_RESTORE_TARGET_ROOT", "").strip()
RESTORE_TARGET_RUN_ID = os.environ.get("NOTEBOOK14_RESTORE_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")).strip()
DEFAULT_RESTORE_TARGET_ROOT = (STRATLAKE_ROOT.parent / "stratlake-notebooks-restored" / (RESTORE_ARCHIVE_ID or "archive") / RESTORE_TARGET_RUN_ID).resolve()
RESTORE_TARGET_ROOT = Path(RESTORE_TARGET_ROOT_OVERRIDE).resolve() if RESTORE_TARGET_ROOT_OVERRIDE else (DEFAULT_RESTORE_TARGET_ROOT if DRIVE_NEEDED or RUN_ARCHIVE_RESTORE else None)
RESTORED_ARTIFACT_ROOT = RESTORE_TARGET_ROOT / "artifacts" if RESTORE_TARGET_ROOT is not None else None

SELECTED_RUN_ID = os.environ.get("NOTEBOOK14_SELECTED_RUN_ID", "").strip()
SELECTED_CATALOG_ID = os.environ.get("NOTEBOOK14_SELECTED_CATALOG_ID", "").strip()
REVIEW_ID = os.environ.get("NOTEBOOK14_REVIEW_ID", "").strip()

for path in [STRATLAKE_ROOT, NOTEBOOKS_ROOT, CONFIG_ROOT, DOCS_ROOT, CONTRACTS_ROOT, ARTIFACT_ROOT, NOTEBOOK14_RUNTIME_ROOT, SUMMARY_OUTPUT_DIR]:
    ensure_dir(path)
if RESTORE_TARGET_ROOT is not None:
    ensure_dir(RESTORE_TARGET_ROOT.parent)

runtime_controls = {
    "profile": NOTEBOOK14_TEST_PROFILE,
    "source_safe_default": NOTEBOOK14_TEST_PROFILE == "evidence_governance_preview",
    "profile_requests_stratlake_init": PROFILE["init"],
    "profile_requires_drive": PROFILE["drive"],
    "drive_needed": DRIVE_NEEDED,
    "allow_drive_mount": ALLOW_DRIVE_MOUNT,
    "run_stratlake_init": RUN_STRATLAKE_INIT,
    "run_archive_restore": RUN_ARCHIVE_RESTORE,
    "run_evidence_review_pack_build": RUN_EVIDENCE_REVIEW_PACK_BUILD,
    "run_evidence_review_pack_validate": RUN_EVIDENCE_REVIEW_PACK_VALIDATE,
    "run_promotion_governance_report": RUN_PROMOTION_GOVERNANCE_REPORT,
    "run_catalog_lineage_export": RUN_CATALOG_LINEAGE_EXPORT,
    "run_archive_checkpoint": RUN_ARCHIVE_CHECKPOINT,
}
path_summary = {
    "stratlake_root": STRATLAKE_ROOT,
    "notebooks_root": NOTEBOOKS_ROOT,
    "config_root": CONFIG_ROOT,
    "docs_root": DOCS_ROOT,
    "contracts_root": CONTRACTS_ROOT,
    "artifact_root": ARTIFACT_ROOT,
    "runtime_root": NOTEBOOK14_RUNTIME_ROOT,
    "drive_root": DRIVE_ROOT,
    "drive_archive_root": DRIVE_ARCHIVE_ROOT,
    "restore_archive_root": RESTORE_ARCHIVE_ROOT,
    "restore_target_root": RESTORE_TARGET_ROOT,
    "restore_target_run_id": RESTORE_TARGET_RUN_ID if RESTORE_TARGET_ROOT is not None else None,
    "restored_artifact_root": RESTORED_ARTIFACT_ROOT,
    "effective_campaign_artifact_root": CAMPAIGN_ARTIFACT_ROOT,
}
display_markdown("### Runtime controls")
display_df(pd.DataFrame([runtime_controls]).T.rename(columns={0: "value"}))
display_markdown("### Workspace paths")
display_df(pd.DataFrame([path_summary]).T.rename(columns={0: "value"}))
display_markdown("### Drive status")
display_df(pd.DataFrame([drive_status]).T.rename(columns={0: "value"}))


## 5. Native command discovery and StratLake initialization


In [ ]:
COMMAND_RESULTS: list[dict[str, Any]] = []
def run_command(command: list[str], *, cwd: Path | None = None, timeout: int = 600, allow_run: bool = False, label: str | None = None) -> dict[str, Any]:
    result: dict[str, Any] = {"label": label or " ".join(command), "command": command, "cwd": cwd.as_posix() if cwd else None, "started_at": utc_now_iso(), "allow_run": allow_run, "returncode": None, "stdout_tail": "", "stderr_tail": "", "duration_seconds": None, "skipped": not allow_run}
    if not allow_run:
        result["completed_at"] = utc_now_iso()
        COMMAND_RESULTS.append(result)
        return result
    started = time.time()
    try:
        completed = subprocess.run(command, cwd=str(cwd) if cwd else None, check=False, capture_output=True, text=True, timeout=timeout)
        result.update({"returncode": completed.returncode, "stdout_tail": tail_text(completed.stdout), "stderr_tail": tail_text(completed.stderr), "duration_seconds": round(time.time() - started, 3), "skipped": False, "completed_at": utc_now_iso()})
    except Exception as exc:
        result.update({"returncode": -1, "stderr_tail": f"{type(exc).__name__}: {exc}", "duration_seconds": round(time.time() - started, 3), "skipped": False, "completed_at": utc_now_iso()})
    COMMAND_RESULTS.append(result)
    return result

def command_results_dataframe() -> pd.DataFrame:
    return pd.DataFrame([{"label": r["label"], "command": " ".join(r["command"]), "returncode": r["returncode"], "skipped": r["skipped"], "duration_seconds": r["duration_seconds"], "stdout_tail_present": bool(r["stdout_tail"]), "stderr_tail_present": bool(r["stderr_tail"])} for r in COMMAND_RESULTS])

NATIVE_COMMANDS = ["stratlake-init-notebook", "stratlake-init-session", "stratlake-notebook-doctor", "stratlake-session-archive-restore-bootstrap", "stratlake-build-evidence-review", "stratlake-run-promotion-governance-report", "stratlake-catalog-index", "stratlake-query-catalog", "stratlake-explore-catalog-evidence", "stratlake-export-catalog-lineage", "stratlake-session-archive-bootstrap"]
command_inventory = []
for command in NATIVE_COMMANDS:
    available = command_available(command)
    command_inventory.append({"command": command, "available": available, "path": shutil.which(command)})
    if available:
        result = run_command([command, "--help"], cwd=STRATLAKE_ROOT, timeout=120, allow_run=True, label=f"{command} --help")
        if result["returncode"] != 0:
            CAVEATS.append(f"Native command help returned non-zero status: {command}")
    else:
        CAVEATS.append(f"Native command unavailable: {command}")

init_result = None
doctor_result = None
if PROFILE["init"] and not RUN_STRATLAKE_INIT:
    CAVEATS.append("Profile requests StratLake workspace init, but NOTEBOOK14_ALLOW_STRATLAKE_INIT=true and RUN_STRATLAKE_INIT=true were not both set.")
if RUN_STRATLAKE_INIT:
    if command_available("stratlake-init-notebook"):
        init_result = run_command(["stratlake-init-notebook", "--root", STRATLAKE_ROOT.as_posix()], cwd=STRATLAKE_ROOT, timeout=600, allow_run=True, label="stratlake init notebook workspace")
        if init_result["returncode"] != 0:
            CAVEATS.append("StratLake workspace initialization returned non-zero status.")
    else:
        CAVEATS.append("StratLake init requested but stratlake-init-notebook is unavailable.")
    for required_dir in [NOTEBOOKS_ROOT, CONFIG_ROOT, DOCS_ROOT, CONTRACTS_ROOT, ARTIFACT_ROOT]:
        if not required_dir.exists():
            CAVEATS.append(f"Expected StratLake workspace directory missing after init: {required_dir}")
    if command_available("stratlake-notebook-doctor"):
        doctor_result = run_command(["stratlake-notebook-doctor", "--root", STRATLAKE_ROOT.as_posix()], cwd=STRATLAKE_ROOT, timeout=600, allow_run=True, label="stratlake notebook doctor")

command_inventory_df = pd.DataFrame(command_inventory)
command_results_df = command_results_dataframe()
display_markdown("### Native command inventory")
display_df(command_inventory_df, max_rows=20)
display_markdown("### Command result summary after command discovery/init")
display_df(command_results_df, max_rows=30)


## 6. Optional restore, restored feature/QA inventory, governance, lineage, and handoff

Archive restore uses `--archive-root <specific archive path>` and `--target-root <isolated restore target>`. Restored feature-run and QA-feature artifacts are classified separately from campaign/evidence/governance artifacts to preserve conservative non-claim boundaries.


In [ ]:
restore_result = None
restore_command = None
if RUN_ARCHIVE_RESTORE:
    if RESTORE_ARCHIVE_ROOT is None:
        CAVEATS.append("Archive restore requested but no concrete restore archive root could be resolved. Set NOTEBOOK14_RESTORE_ARCHIVE_ID or NOTEBOOK14_RESTORE_ARCHIVE_ROOT.")
    elif str(RESTORE_ARCHIVE_ROOT).startswith(str(DRIVE_MOUNT_POINT)) and not drive_status.get("mounted_after", False):
        CAVEATS.append("Archive restore requested from Drive, but Drive is not mounted.")
    elif not RESTORE_ARCHIVE_ROOT.exists():
        CAVEATS.append(f"Archive restore root does not exist: {RESTORE_ARCHIVE_ROOT}")
    elif RESTORE_TARGET_ROOT is None:
        CAVEATS.append("Archive restore requested but no isolated restore target root could be resolved.")
    elif RESTORE_TARGET_ROOT.exists() and any(RESTORE_TARGET_ROOT.rglob("*")) and os.environ.get("NOTEBOOK14_ALLOW_NONEMPTY_RESTORE_TARGET", "false").lower() != "true":
        CAVEATS.append(f"Archive restore target root is not empty; refusing to restore without NOTEBOOK14_ALLOW_NONEMPTY_RESTORE_TARGET=true: {RESTORE_TARGET_ROOT}")
    elif command_available("stratlake-session-archive-restore-bootstrap"):
        restore_command = ["stratlake-session-archive-restore-bootstrap", "--archive-root", RESTORE_ARCHIVE_ROOT.as_posix(), "--target-root", RESTORE_TARGET_ROOT.as_posix()]
        restore_result = run_command(restore_command, cwd=STRATLAKE_ROOT, timeout=900, allow_run=True, label="restore Notebook 13 session archive")
        if restore_result["returncode"] == 0 and RESTORED_ARTIFACT_ROOT is not None and RESTORED_ARTIFACT_ROOT.exists():
            CAMPAIGN_ARTIFACT_ROOT = RESTORED_ARTIFACT_ROOT
            path_summary["effective_campaign_artifact_root"] = CAMPAIGN_ARTIFACT_ROOT
            path_summary["restore_completed"] = True
        elif restore_result["returncode"] == 0:
            CAVEATS.append("Archive restore returned zero but restored artifact root was not found.")
            path_summary["restore_completed"] = True
        else:
            CAVEATS.append("Archive restore command returned non-zero status.")
            path_summary["restore_completed"] = False
    else:
        CAVEATS.append("Archive restore requested but restore command is unavailable.")
else:
    CAVEATS.append("Archive restore not run; Notebook 14 is using local/runtime artifact discovery only.")

restore_diagnostic_rows = []
if restore_result is None:
    restore_diagnostic_rows.append({"label": "restore Notebook 13 session archive", "executed": False, "returncode": None, "archive_root": RESTORE_ARCHIVE_ROOT.as_posix() if RESTORE_ARCHIVE_ROOT else None, "target_root": RESTORE_TARGET_ROOT.as_posix() if RESTORE_TARGET_ROOT else None, "archive_root_exists": RESTORE_ARCHIVE_ROOT.exists() if RESTORE_ARCHIVE_ROOT else False, "stdout_tail_present": False, "stderr_tail_present": False})
else:
    restore_diagnostic_rows.append({"label": restore_result.get("label"), "executed": not restore_result.get("skipped", False), "returncode": restore_result.get("returncode"), "archive_root": RESTORE_ARCHIVE_ROOT.as_posix() if RESTORE_ARCHIVE_ROOT else None, "target_root": RESTORE_TARGET_ROOT.as_posix() if RESTORE_TARGET_ROOT else None, "archive_root_exists": RESTORE_ARCHIVE_ROOT.exists() if RESTORE_ARCHIVE_ROOT else False, "stdout_tail_present": bool(restore_result.get("stdout_tail")), "stderr_tail_present": bool(restore_result.get("stderr_tail"))})
restore_diagnostics_df = pd.DataFrame(restore_diagnostic_rows)
display_markdown("### Archive restore command diagnostics")
display_df(restore_diagnostics_df, max_rows=10)
if restore_command is not None:
    display_markdown("### Archive restore command")
    print(" ".join(restore_command))
if restore_result is not None and restore_result.get("stderr_tail"):
    display_markdown("### Archive restore stderr tail")
    print(restore_result.get("stderr_tail", ""))
if restore_result is not None and restore_result.get("stdout_tail"):
    display_markdown("### Archive restore stdout tail")
    print(restore_result.get("stdout_tail", ""))

post_optional_command_results_df = command_results_dataframe()
display_markdown("### Command result summary after optional restore/review/governance/catalog steps")
display_df(post_optional_command_results_df.tail(20), max_rows=20)

POST_RESTORE_INVENTORY_LIMIT = int(os.environ.get("NOTEBOOK14_POST_RESTORE_INVENTORY_LIMIT", "200"))
def bounded_file_inventory(root: Path | None, *, scope: str, limit: int = POST_RESTORE_INVENTORY_LIMIT) -> tuple[pd.DataFrame, dict[str, Any]]:
    if root is None or not root.exists():
        return pd.DataFrame(), {"scope": scope, "root": root.as_posix() if root else None, "exists": False, "total_files": 0, "total_bytes": 0, "returned_rows": 0, "suffix_counts": {}, "top_dir_counts": {}}
    rows = []
    total_files = 0
    total_bytes = 0
    suffix_counts: dict[str, int] = {}
    top_dir_counts: dict[str, int] = {}
    for path in sorted((p for p in root.rglob("*") if p.is_file()), key=lambda p: p.as_posix()):
        total_files += 1
        stat = path.stat()
        total_bytes += stat.st_size
        suffix = path.suffix or "<none>"
        suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
        try:
            rel = path.relative_to(root)
            rel_path = rel.as_posix()
            top_dir = rel.parts[0] if len(rel.parts) > 1 else "<root>"
        except ValueError:
            rel_path = path.as_posix()
            top_dir = "<external>"
        top_dir_counts[top_dir] = top_dir_counts.get(top_dir, 0) + 1
        if len(rows) < limit:
            rows.append({"scope": scope, "root": root.as_posix(), "relative_path": rel_path, "path": path.as_posix(), "top_dir": top_dir, "suffix": suffix, "size_bytes": stat.st_size, "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat()})
    summary = {"scope": scope, "root": root.as_posix(), "exists": True, "total_files": total_files, "total_bytes": total_bytes, "returned_rows": len(rows), "suffix_counts": dict(sorted(suffix_counts.items())), "top_dir_counts": dict(sorted(top_dir_counts.items()))}
    return pd.DataFrame(rows), summary

post_restore_inventory_frames = []
post_restore_inventory_summaries = []
for scope, root in [("restore_target_root", RESTORE_TARGET_ROOT), ("restored_artifact_root", RESTORED_ARTIFACT_ROOT)]:
    frame, summary = bounded_file_inventory(root, scope=scope)
    post_restore_inventory_summaries.append(summary)
    if not frame.empty:
        post_restore_inventory_frames.append(frame)
post_restore_file_inventory_df = pd.concat(post_restore_inventory_frames, ignore_index=True) if post_restore_inventory_frames else pd.DataFrame()
post_restore_inventory_summary_df = pd.DataFrame(post_restore_inventory_summaries)
display_markdown("### Post-restore file inventory summary")
display_df(post_restore_inventory_summary_df, max_rows=10)
if not post_restore_file_inventory_df.empty:
    display_markdown("### Bounded post-restore file inventory")
    display_df(post_restore_file_inventory_df, max_rows=50)

RESTORED_FEATURE_QA_PATTERNS = {
    "restored_feature_run_summary": ["feature_runs/**/*summary*.json", "feature_runs/**/*summary*.csv"],
    "restored_feature_metadata": ["features/**/*metadata*.json", "features/**/*metadata*.csv", "features/**/*.json"],
    "restored_feature_qa_summary": ["qa/features/**/*.json", "qa/features/**/*.csv"],
    "restored_session_archive_report": ["_derived/session_archives/**/*.json"],
}
def discover_pattern_files(root: Path, patterns: list[str], limit: int = 200) -> list[Path]:
    if not root.exists():
        return []
    found: list[Path] = []
    for pattern in patterns:
        found.extend(p for p in root.glob(pattern) if p.is_file())
    return sorted(set(found), key=lambda p: p.as_posix())[:limit]

restored_feature_qa_rows = []
if RESTORED_ARTIFACT_ROOT is not None:
    for artifact_type, patterns in RESTORED_FEATURE_QA_PATTERNS.items():
        for path in discover_pattern_files(RESTORED_ARTIFACT_ROOT, patterns):
            stat = path.stat()
            restored_feature_qa_rows.append({
                "artifact_type": artifact_type,
                "path": path.as_posix(),
                "relative_path": path.relative_to(RESTORED_ARTIFACT_ROOT).as_posix(),
                "name": path.name,
                "suffix": path.suffix,
                "size_bytes": stat.st_size,
                "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
                "classification": "restored_feature_or_qa_artifact_non_governance_non_promotion",
            })
restored_feature_qa_artifacts_df = pd.DataFrame(restored_feature_qa_rows)
if not restored_feature_qa_artifacts_df.empty:
    display_markdown("### Restored feature/QA artifacts classified separately")
    display_df(restored_feature_qa_artifacts_df, max_rows=50)

ARTIFACT_PATTERNS = {
    "campaign_manifest": ["**/campaign*/**/manifest.json", "**/*campaign*manifest*.json"],
    "campaign_summary": ["**/*campaign*summary*.json", "**/*campaign*summary*.csv", "**/*campaign*report*.md"],
    "run_registry": ["**/*run*registry*.json", "**/*registry*.csv", "**/registry*.json"],
    "metrics": ["**/*metrics*.json", "**/*metrics*.csv"],
    "split_metrics": ["**/*split*metrics*.json", "**/*split*metrics*.csv"],
    "promotion_gate": ["**/*promotion*gate*.json", "**/*promotion*gate*.csv"],
    "governance": ["**/*governance*.json", "**/*governance*.csv", "**/*governance*.md"],
    "evidence_review": ["**/_derived/evidence_review/**/*.json", "**/_derived/evidence_review/**/*.csv", "**/_derived/evidence_review/**/*.md"],
}
EXCLUDED_ARTIFACT_PATH_PARTS = {"_notebook_14_runtime", "_notebook_runtime"}
EXCLUDED_ARTIFACT_NAME_PREFIXES = ("notebook_14_", "notebook14_")
def is_notebook_runtime_artifact_path(path: Path) -> bool:
    return bool(set(path.parts) & EXCLUDED_ARTIFACT_PATH_PARTS) or path.name.startswith(EXCLUDED_ARTIFACT_NAME_PREFIXES)
def is_reviewable_artifact_path(path: Path) -> bool:
    return path.is_file() and not is_notebook_runtime_artifact_path(path)
def discover_files(root: Path, patterns: list[str], limit: int = 200) -> list[Path]:
    if not root.exists():
        return []
    found = []
    for pattern in patterns:
        found.extend(path for path in root.glob(pattern) if is_reviewable_artifact_path(path))
    return sorted(set(found), key=lambda p: p.as_posix())[:limit]
def discover_notebook_runtime_outputs(root: Path, limit: int = 200) -> list[Path]:
    if not root.exists():
        return []
    return sorted({p for p in root.rglob("*") if p.is_file() and is_notebook_runtime_artifact_path(p)}, key=lambda p: p.as_posix())[:limit]

artifact_rows = []
for artifact_type, patterns in ARTIFACT_PATTERNS.items():
    for path in discover_files(CAMPAIGN_ARTIFACT_ROOT, patterns):
        stat = path.stat()
        artifact_rows.append({"artifact_type": artifact_type, "path": path.as_posix(), "name": path.name, "suffix": path.suffix, "size_bytes": stat.st_size, "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(), "classification": "reviewable_native_or_upstream_artifact_candidate"})
runtime_artifact_rows = []
for path in discover_notebook_runtime_outputs(CAMPAIGN_ARTIFACT_ROOT):
    stat = path.stat()
    runtime_artifact_rows.append({"artifact_type": "notebook_14_runtime_output", "path": path.as_posix(), "name": path.name, "suffix": path.suffix, "size_bytes": stat.st_size, "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(), "classification": "notebook_runtime_non_canonical_excluded_from_reviewable_artifacts"})
artifact_inventory_df = pd.DataFrame(artifact_rows)
notebook_runtime_inventory_df = pd.DataFrame(runtime_artifact_rows)

if artifact_inventory_df.empty:
    CAVEATS.append(f"No campaign/evidence/governance artifacts discovered under {CAMPAIGN_ARTIFACT_ROOT}. Notebook runtime outputs were excluded from reviewable artifact discovery.")
    restored_artifact_summary = next((row for row in post_restore_inventory_summaries if row.get("scope") == "restored_artifact_root"), {})
    restored_file_count = int(restored_artifact_summary.get("total_files", 0) or 0)
    if restore_result is not None and restore_result.get("returncode") == 0 and not restored_feature_qa_artifacts_df.empty:
        CAVEATS.append(f"Restored artifact root contains {len(restored_feature_qa_artifacts_df)} feature/QA/session-restore artifacts classified separately; these do not imply promotion governance readiness.")
    elif restore_result is not None and restore_result.get("returncode") == 0 and restored_file_count > 0:
        CAVEATS.append(f"Restored artifact root contains {restored_file_count} files, but none matched current campaign/evidence/governance or feature/QA discovery patterns; inspect post-restore inventory before widening patterns.")
    elif restore_result is not None and restore_result.get("returncode") == 0:
        CAVEATS.append("Archive restore completed, but restored artifact root contains no files to classify.")
else:
    display_df(artifact_inventory_df, max_rows=50)
if not notebook_runtime_inventory_df.empty:
    display_markdown("### Notebook runtime outputs excluded from reviewable artifact discovery")
    display_df(notebook_runtime_inventory_df, max_rows=20)

evidence_build_result = evidence_validate_result = governance_result = None
if RUN_EVIDENCE_REVIEW_PACK_BUILD:
    if not SELECTED_RUN_ID and not SELECTED_CATALOG_ID:
        CAVEATS.append("Evidence review build requested but no selected run/catalog id is set.")
    else:
        cmd = ["stratlake-build-evidence-review", "build", "--artifacts-root", ARTIFACT_ROOT.as_posix()]
        if SELECTED_RUN_ID:
            cmd += ["--selected-run-id", SELECTED_RUN_ID]
        if SELECTED_CATALOG_ID:
            cmd += ["--selected-catalog-id", SELECTED_CATALOG_ID]
        if REVIEW_ID:
            cmd += ["--review-id", REVIEW_ID]
        evidence_build_result = run_command(cmd, cwd=STRATLAKE_ROOT, timeout=900, allow_run=command_available("stratlake-build-evidence-review"), label="build derived evidence review pack")
else:
    CAVEATS.append("Native evidence review pack build not run; derived review pack evidence is absent unless previously generated.")
if RUN_EVIDENCE_REVIEW_PACK_VALIDATE:
    cmd = ["stratlake-build-evidence-review", "validate", "--artifacts-root", ARTIFACT_ROOT.as_posix()]
    if REVIEW_ID:
        cmd += ["--review-id", REVIEW_ID]
    evidence_validate_result = run_command(cmd, cwd=STRATLAKE_ROOT, timeout=600, allow_run=command_available("stratlake-build-evidence-review"), label="validate derived evidence review pack")
else:
    CAVEATS.append("Native evidence review pack validation not run.")
if RUN_PROMOTION_GOVERNANCE_REPORT:
    governance_result = run_command(["stratlake-run-promotion-governance-report", "--artifact-root", ARTIFACT_ROOT.as_posix(), "--output-dir", GOVERNANCE_OUTPUT_DIR.as_posix()], cwd=STRATLAKE_ROOT, timeout=900, allow_run=command_available("stratlake-run-promotion-governance-report"), label="run promotion governance report")
else:
    CAVEATS.append("Native promotion governance report not run; governance readiness is not claimed.")
if RUN_CATALOG_LINEAGE_EXPORT:
    ensure_dir(CATALOG_LINEAGE_OUTPUT_DIR)
    for label, cmd in [
        ("catalog index", ["stratlake-catalog-index", "--artifact-root", ARTIFACT_ROOT.as_posix()]),
        ("query catalog", ["stratlake-query-catalog", "--artifact-root", ARTIFACT_ROOT.as_posix()]),
        ("explore catalog evidence", ["stratlake-explore-catalog-evidence", "--artifact-root", ARTIFACT_ROOT.as_posix()]),
        ("export catalog lineage", ["stratlake-export-catalog-lineage", "--artifact-root", ARTIFACT_ROOT.as_posix(), "--output-dir", CATALOG_LINEAGE_OUTPUT_DIR.as_posix()]),
    ]:
        run_command(cmd, cwd=STRATLAKE_ROOT, timeout=900, allow_run=command_available(cmd[0]), label=label)
else:
    CAVEATS.append("Catalog/lineage export not run.")

EXPECTED_EVIDENCE_REVIEW_FILES = ["manifest.json", "review_request.json", "review_summary.json", "catalog_health_diagnostics.json", "validation.json", "selected_record.json", "related_records.json", "resolver_resolution.json", "evidence_index.json", "artifact_inventory.csv", "report.md"]
EXPECTED_GOVERNANCE_FILES = ["promotion_governance_summary.json", "promotion_outcome_matrix.csv", "reason_code_summary.csv", "severity_summary.csv", "workflow_summary.csv", "consistency_validation.json", "promotion_governance_report.md", "manifest.json"]
def latest_matching_dir(root: Path, required_names: list[str]) -> Path | None:
    if not root.exists():
        return None
    scored = []
    for candidate in [p for p in root.rglob("*") if p.is_dir() and not is_notebook_runtime_artifact_path(p)]:
        score = sum((candidate / name).exists() for name in required_names)
        if score:
            scored.append((score, candidate.stat().st_mtime, candidate))
    return sorted(scored, key=lambda item: (item[0], item[1]), reverse=True)[0][2] if scored else None
latest_review_pack_dir = latest_matching_dir(EVIDENCE_REVIEW_OUTPUT_DIR, EXPECTED_EVIDENCE_REVIEW_FILES)
latest_governance_dir = latest_matching_dir(GOVERNANCE_OUTPUT_DIR, EXPECTED_GOVERNANCE_FILES)
review_pack_df = pd.DataFrame([{"expected_file": name, "found": bool(latest_review_pack_dir and (latest_review_pack_dir / name).exists()), "classification": "derived_non_authoritative_write_back_forbidden"} for name in EXPECTED_EVIDENCE_REVIEW_FILES])
governance_files_df = pd.DataFrame([{"expected_file": name, "found": bool(latest_governance_dir and (latest_governance_dir / name).exists()), "classification": "read_only_governance_observability"} for name in EXPECTED_GOVERNANCE_FILES])
if latest_review_pack_dir is None:
    CAVEATS.append("No derived evidence review pack directory discovered.")
if latest_governance_dir is None:
    CAVEATS.append("No promotion governance report directory discovered.")
display_markdown("### Evidence review pack files")
display_df(review_pack_df, max_rows=20)
display_markdown("### Governance report files")
display_df(governance_files_df, max_rows=20)

NON_CLAIMS = ["production_readiness", "strategy_approval", "promotion_readiness", "governance_readiness", "statistical_significance", "alpha_validation", "split_metric_completeness", "artifact_completeness", "source_runtime_equivalence", "derived_review_pack_is_canonical", "artifact_presence_proves_current_session_execution", "restored_feature_qa_artifacts_imply_governance_readiness"]
if artifact_inventory_df.empty:
    CAVEATS.append("Artifact completeness cannot be assessed because no reviewable campaign artifact inventory was discovered. Notebook runtime outputs were excluded.")
split_metric_rows = artifact_inventory_df[artifact_inventory_df["artifact_type"].eq("split_metrics")] if not artifact_inventory_df.empty else pd.DataFrame()
governance_artifact_rows = artifact_inventory_df[artifact_inventory_df["artifact_type"].eq("governance")] if not artifact_inventory_df.empty else pd.DataFrame()
if split_metric_rows.empty:
    CAVEATS.append("No split-metric artifacts discovered; split-metric completeness is not claimed.")
if governance_artifact_rows.empty and latest_governance_dir is None:
    CAVEATS.append("No reviewable governance artifacts discovered; governance readiness is not claimed. Restored feature/QA artifacts are classified separately and do not imply governance readiness.")
seen = set()
CAVEATS = [c for c in CAVEATS if not (c in seen or seen.add(c))]
audit_register = {"generated_at": utc_now_iso(), "profile": NOTEBOOK14_TEST_PROFILE, "caveat_count": len(CAVEATS), "caveats": CAVEATS, "non_claims": NON_CLAIMS, "drive_status": drive_status, "restore_diagnostics": restore_diagnostics_df.to_dict(orient="records"), "governance_boundary": "read_only_observability_not_promotion_decision", "notebook_runtime_outputs_excluded": True, "restored_feature_qa_artifacts_classified_separately": True}
audit_register


## 7. Runtime handoff summary


In [ ]:
handoff_summary = {
    "notebook": "Notebook 14 — StratLake Campaign Evidence Review Pack and Governance Audit",
    "stance": "notebook_14_staged_cleaned_source_safe_evidence_governance_audit",
    "generated_at": utc_now_iso(),
    "profile": NOTEBOOK14_TEST_PROFILE,
    "runtime_controls": runtime_controls,
    "paths": path_summary,
    "drive_status": drive_status,
    "command_inventory": command_inventory_df.to_dict(orient="records") if not command_inventory_df.empty else [],
    "command_results": COMMAND_RESULTS,
    "restore_diagnostics": restore_diagnostics_df.to_dict(orient="records"),
    "post_restore_inventory_summary": post_restore_inventory_summary_df.to_dict(orient="records") if not post_restore_inventory_summary_df.empty else [],
    "post_restore_inventory_rows": len(post_restore_file_inventory_df),
    "restored_feature_qa_artifact_rows": len(restored_feature_qa_artifacts_df),
    "restored_feature_qa_artifacts": restored_feature_qa_artifacts_df.to_dict(orient="records") if not restored_feature_qa_artifacts_df.empty else [],
    "artifact_inventory_rows": len(artifact_inventory_df),
    "notebook_runtime_output_rows": len(notebook_runtime_inventory_df),
    "notebook_runtime_outputs_excluded_from_reviewable_artifacts": True,
    "latest_review_pack_dir": latest_review_pack_dir.as_posix() if latest_review_pack_dir else None,
    "review_pack_file_status": review_pack_df.to_dict(orient="records"),
    "latest_governance_dir": latest_governance_dir.as_posix() if latest_governance_dir else None,
    "governance_file_status": governance_files_df.to_dict(orient="records"),
    "caveats": CAVEATS,
    "non_claims": NON_CLAIMS,
}
summary_path = SUMMARY_OUTPUT_DIR / "notebook_14_evidence_governance_handoff_summary.json"
commands_path = SUMMARY_OUTPUT_DIR / "notebook_14_command_results.json"
inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_artifact_inventory.csv"
runtime_inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_runtime_output_inventory.csv"
post_restore_inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_post_restore_file_inventory.csv"
post_restore_summary_path = SUMMARY_OUTPUT_DIR / "notebook_14_post_restore_inventory_summary.json"
restored_feature_qa_inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_restored_feature_qa_artifact_inventory.csv"
caveats_path = SUMMARY_OUTPUT_DIR / "notebook_14_caveats.json"
write_json(summary_path, handoff_summary)
write_json(commands_path, COMMAND_RESULTS)
write_json(caveats_path, audit_register)
write_json(post_restore_summary_path, post_restore_inventory_summaries)
if not artifact_inventory_df.empty:
    write_dataframe_csv(inventory_path, artifact_inventory_df)
if not notebook_runtime_inventory_df.empty:
    write_dataframe_csv(runtime_inventory_path, notebook_runtime_inventory_df)
if not post_restore_file_inventory_df.empty:
    write_dataframe_csv(post_restore_inventory_path, post_restore_file_inventory_df)
if not restored_feature_qa_artifacts_df.empty:
    write_dataframe_csv(restored_feature_qa_inventory_path, restored_feature_qa_artifacts_df)
{
    "summary_path": summary_path.as_posix(),
    "commands_path": commands_path.as_posix(),
    "inventory_path": inventory_path.as_posix() if not artifact_inventory_df.empty else None,
    "runtime_inventory_path": runtime_inventory_path.as_posix() if not notebook_runtime_inventory_df.empty else None,
    "post_restore_inventory_path": post_restore_inventory_path.as_posix() if not post_restore_file_inventory_df.empty else None,
    "post_restore_summary_path": post_restore_summary_path.as_posix(),
    "restored_feature_qa_inventory_path": restored_feature_qa_inventory_path.as_posix() if not restored_feature_qa_artifacts_df.empty else None,
    "caveats_path": caveats_path.as_posix(),
}


## 8. Final conservative handoff

Notebook 14 is source-safe when committed with no outputs, execution counts are null, the default profile remains `evidence_governance_preview`, restore/evidence/governance/catalog/checkpoint gates are disabled by default, Drive mounting is explicit and gated, `stratlake-init-notebook` runs against an explicit workspace root when enabled, archive restore diagnostics use `--archive-root`/`--target-root`, restore targets are isolated from the initialized workspace by default, restored artifact discovery switches to the restored target only after successful restore, restored feature/QA artifacts are classified separately from campaign/evidence/governance artifacts, bounded post-restore inventories distinguish missing artifacts from pattern-mismatch cases, notebook runtime outputs are excluded from campaign/evidence/governance artifact discovery, and final claims remain conservative.
